# Week 7 Lab: Complete IV and 2SLS Estimation

## Learning Objectives

By the end of this lab, you will be able to:
1. Implement OLS, IV, and 2SLS estimation with proper standard errors
2. Test for instrument relevance using first-stage F-statistics
3. Understand weak instrument problems
4. Compare estimation results across different methods

## Overview

This notebook builds upon Week 6, providing complete implementations of:

1. **OLS with robust standard errors**
2. **IV estimation** (just-identified case: one instrument per endogenous variable)
3. **2SLS estimation** (over-identified case: more instruments than endogenous variables)
4. **First-stage F-test** for weak instruments

We'll apply these to Card's (1993) return to schooling estimation.

In [ ]:
using Downloads, DelimitedFiles, LinearAlgebra, Printf, Statistics, Distributions

## Exercise 1: Load and Prepare Data

In [ ]:
# Load Card's (1993) data
url = "https://raw.githubusercontent.com/juergenmeinecke/EMET8014/main/datasets/card.csv"
data = readdlm(Downloads.download(url), ',')

# Dependent variable: log wages
Y = Vector{Float64}(data[:, 33])
N = length(Y)

# Construct regressor matrix X
const_term = ones(N)          # Column 1: constant
educ = data[:, 4]             # Column 2: education (ENDOGENOUS)
exper = data[:, 32]           # Column 3: experience
expersq = data[:, 34] ./ 100  # Column 4: experience^2/100
black = data[:, 22]           # Column 5: black dummy
south = data[:, 24]           # Column 6: south dummy
smsa = data[:, 23]            # Column 7: metropolitan area
smsa66 = data[:, 25]          # Column 8: metro area in 1966
region = data[:, 12:19]       # Columns 9-16: regional dummies

X = Matrix{Float64}(hcat(const_term, educ, exper, expersq, black, south, smsa, smsa66, region))
N, K = size(X)

println("Sample size: N = $N")
println("Number of regressors: K = $K")

## Exercise 2: Creating an EMET8014 Functions Library

Your notebook folder contains a file `emet_8014_functions.jl`. It contains a collection of functions which you will be using over the next two weeks. More specifically, it only contains the **scaffolding** for these functions, you will need to fill in the details!

Start by adding last week's function `lm_ols` at the very top of `emet_8014_functions.jl` (just copy and paste). 



In [ ]:
# here's how you can 'load' the lm_ols function now:
include("emet_8014_functions.jl");

# now you have access to lm_ols

## Exercise 3: Inference Function

In your file `emet_8014_functions.jl` you will find the scaffolding for a new function `lm_inference`. This function
takes an estimator and its covariance matrix as arguments and returns

* a vector containing the standard errors;

* a vector containing the t-statistics;

* a vector containing the p-values (two-sided, using the normal distribution).

Work inside `emet_8014_functions.jl` to complete the function `lm_inference`.

## Exercise 4: OLS Estimation

Now use your two new functions to run OLS and the associated hypothesis tests.

In [ ]:
# OLS estimation
β_ols, Omega_ols = nothing # YOUR CODE HERE
se_ols, t_ols, p_ols = nothing # YOUR CODE HERE

@printf "OLS Results for Return to Schooling (β2):\n"
@printf "  Point estimate: %.4f\n" β_ols[2]
@printf "  Robust SE:      %.4f\n" se_ols[2]
@printf "  t-statistic:     %.2f\n" t_ols[2]
@printf "  p-value    :     %.3f\n" p_ols[2]

## Instrumental Variables Setup

### Notation

Following standard notation:
- $X = (X_1, X_2)$ where $X_1$ is exogenous and $X_2$ is endogenous
- $Z = (Z_1, Z_2)$ where $Z_1 = X_1$ and $Z_2$ are excluded instruments

### Card's Instruments

1. **nearc4**: Lives near 4-year college
2. **nearc2**: Lives near 2-year college

In [ ]:
# Partition regressors and define instruments
X1 = Matrix{Float64}(X[:, [1; 3:K]])        # Exogenous: constant, exper, expersq, controls
X2 = X[:, 2]                                # Endogenous: education

Z1 = X1                                     # Exogenous vars are their own instruments
Z2_nearc4 = Vector{Float64}(data[:, 3])     # nearc4 only
Z2_both = Matrix{Float64}(data[:, [2, 3]])  # nearc2 and nearc4

println("Dimensions:")
println("  X1: $(size(X1)) - Exogenous regressors")
println("  X2: $(length(X2)) - Endogenous regressor")
println("  Z2 (nearc4 only): $(length(Z2_nearc4))")
println("  Z2 (both IVs): $(size(Z2_both))")

## Exercise 5: First-Stage Regression

The **first-stage regression** examines instrument relevance:
$$\text{educ}_i = Z_i' \pi + v_i$$

We're interested in the coefficient(s) on the excluded instruments $Z_2$.

In [ ]:
# First stage with nearc4 only
Z_iv = Matrix{Float64}(hcat(Z1, Z2_nearc4))

π_hat, Omega_π = nothing # YOUR CODE HERE
se_π, t_π, p_π = nothing # YOUR CODE HERE

@printf "\nFirst-Stage Regression (IV with nearc4 only):\n"
@printf "  Coefficient on nearc4: %.3f\n" π_hat[end]
@printf "  Robust SE:             %.3f\n" se_π[end]
@printf "  t-statistic:            %.2f\n" t_π[end]
@printf "  p-value:               %.3f\n" p_π[end]
@printf "\nInterpretation: Living near a 4-year college increases\n"
@printf "education by approximately %.2f years.\n" π_hat[end]

## Exercise 6: IV Estimation Function

For the **just-identified** case (one instrument per endogenous variable), the IV estimator is:

$$\hat{\beta}^{IV} = (Z'X)^{-1} Z'Y$$

### The Asymptotic Distribution of IV

Under standard assumptions (but allowing for heteroskedasticity):

$$\sqrt{N}(\hat{\beta}^{\text{IV}} - \beta) \xrightarrow{d} N(0, \Omega)$$

where the **sandwich formula** for $\Omega$ is:

$$\Omega = E(Z_i X_i')^{-1} \cdot E(u_i^2 Z_i Z_i') \cdot E(X_i Z_i')^{-1}$$



### Estimation

We estimate $\Omega$ using the sample analog with a degrees-of-freedom correction:

$$\begin{aligned}
    \hat{\Omega}_{IV}
        :=& \left(\tfrac{1}{N} \sum_{i=1}^N Z_i X_i'\right)^{-1} 
            \left(\tfrac{1}{N-K} \sum_{i=1}^N \hat{u}_i^2 Z_i Z_i'\right) 
            \left(\tfrac{1}{N} \sum_{i=1}^N X_i Z_i'\right)^{-1} \\
        =& \left(\tfrac{1}{N} Z'X\right)^{-1} 
            \left(\tfrac{1}{N-K} Z' D Z\right) 
            \left(\tfrac{1}{N} X'Z\right)^{-1}\\       
        =& \tfrac{N^2}{N-K} \left(Z'X\right)^{-1} 
            \left(Z' D Z\right) 
            \left(X'Z\right)^{-1}\\
\text{where } D &:= \text{diag}(\hat{u}_1^2, \ldots, \hat{u}_N^2)
\end{aligned}$$

<div class="alert alert-success">

**Key Result:** 

The variance of $\hat{\beta}^{\text{IV}}$ is then approximately $\hat{\Omega}_{IV}/N = \frac{N}{N-K}(Z'X)^{-1} Z' D Z (X'Z)^{-1}$.
    
</div>

Use the function template for `lm_iv` which you can find in `emet_8014_functions.jl`. Write `lm_iv` such that it takes the arguments `Y` and `X1`, `X2`, `Z1`, and `Z2` and returns

* a vector containing the IV estimator;

* a matrix containing the estimated asymptotic covariance of $\widehat{\beta}^{IV}$.

## Exercise 7: IV Estimation Results

In [ ]:
# IV estimation with nearc4
β_iv, Omega_iv = nothing # YOUR CODE HERE
se_iv, t_iv, p_iv = nothing # YOUR CODE HERE

@printf "\nIV Results (using nearc4 as instrument):\n"
@printf "  Return to schooling (β2): %.3f\n" β_iv[end]
@printf "  Robust SE:                %.3f\n" se_iv[end]
@printf "  t-statistic:               %.2f\n" t_iv[end]
@printf "  p-value:                  %.3f\n" p_iv[end]

## Exercise 8: 2SLS Estimation

When we have **more instruments than endogenous variables** (over-identified case), we use 2SLS:

$$\hat{\beta}^{2SLS} = (X'P_Z X)^{-1} X'P_Z Y$$

where $P_Z = Z(Z'Z)^{-1}Z'$ is the projection matrix onto Z.


### The Asymptotic Distribution of 2SLS

Under standard assumptions (but allowing for heteroskedasticity):

$$\sqrt{N}(\hat{\beta}^{\text{2SLS}} - \beta) \xrightarrow{d} N(0, \Omega)$$

where the **sandwich formula** for $\Omega$ is:

                                                                                         
$$\begin{aligned}
\Omega 
    &= (C_{xz} C_{zz}^{-1} C_{zx})^{-1} 
        C_{xz} C_{zz}^{-1} 
        E(u_i^2 Z_i Z_i') 
        C_{zz}^{-1} C_{zx} 
        (C_{xz} C_{zz}^{-1} C_{zx})^{-1}
\end{aligned}$$
 
 
where 
$C_{xz} = E (X_i Z_i')$, and $C_{zz} = E (Z_i Z_i')$, and $C_{zx} = E (Z_i X_i')$.


### Estimation

We estimate $\Omega$ using the sample analog (no degrees of freedom correction):

$$\begin{aligned}
\hat{\Omega}_{2SLS}
    :=& (\hat{C}_{xz} \hat{C}_{zz}^{-1} \hat{C}_{zx})^{-1} 
        \hat{C}_{xz} \hat{C}_{zz}^{-1} 
        \left(\tfrac{1}{N-K} \sum_{i=1}^N \hat{u}_i^2 Z_i Z_i'\right) 
        \hat{C}_{zz}^{-1} \hat{C}_{zx} 
        (\hat{C}_{xz} \hat{C}_{zz}^{-1} \hat{C}_{zx})^{-1}\\
    =& (\hat{C}_{xz} \hat{C}_{zz}^{-1} \hat{C}_{zx})^{-1} 
        \hat{C}_{xz} \hat{C}_{zz}^{-1} 
        \left( \tfrac{1}{N-K} Z' D Z\right) 
        \hat{C}_{zz}^{-1} \hat{C}_{zx} 
        (\hat{C}_{xz} \hat{C}_{zz}^{-1} \hat{C}_{zx})^{-1}\\
    =& \tfrac{N^2}{N-K} (X'P_Z X)^{-1} (X'P_Z D P_Z X) (X'P_Z X)^{-1}\\
\text{where } D &:= \text{diag}(\hat{u}_1^2, \ldots, \hat{u}_N^2)    
\end{aligned}$$


<div class="alert alert-success">

**Key Result:** 

The variance of $\hat{\beta}^{\text{2SLS}}$ is then approximately 
$\hat{\Omega}_{2SLS}/N = \frac{N}{N-K}(X'P_Z X)^{-1} (X'P_Z D P_Z X) (X'P_Z X)^{-1}$.
    
</div>



Work on the function template `lm_2sls` so that it takes the arguments `Y` and `X1`, `X2`, `Z1`, and `Z2` and returns

* a vector containing the 2SLS estimates;

* a matrix containing the estimated asymptotic covariance matrix of $\widehat{\beta}^{2SLS}$.

## Exercise 9: 2SLS with Both Instruments

In [ ]:
# 2SLS with both nearc2 and nearc4
β_2sls, Omega_2sls = nothing # YOUR CODE HERE
se_2sls, t_2sls, p_2sls = nothing # YOUR CODE HERE

@printf "\n2SLS Results (using nearc2 and nearc4):\n"
@printf "  Return to schooling (β2): %.3f\n" β_2sls[end]
@printf "  Robust SE:                 %.3f\n" se_2sls[end]
@printf "  t-statistic:               %.2f\n" t_2sls[end]
@printf "  p-value:                  %.3f\n" p_2sls[end]

## Exercise 10: First-Stage with Both Instruments

In [ ]:
# First stage with both instruments
Z_2sls = Matrix{Float64}(hcat(Z1, Z2_both))

π_hat_2sls, Omega_π_2sls = nothing # YOUR CODE HERE
se_π_2sls, t_π_2sls, p_π_2sls = nothing # YOUR CODE HERE

@printf "\nFirst-Stage Regression (with both instruments):\n"
@printf "  nearc2 coefficient: %.2f (SE: %.2f, t: %.2f)\n" π_hat_2sls[end-1] se_π_2sls[end-1] t_π_2sls[end-1]
@printf "  nearc4 coefficient: %.2f (SE: %.2f, t: %.2f)\n" π_hat_2sls[end] se_π_2sls[end] t_π_2sls[end]

## Exercise 11: First-Stage F-Test

### The Weak Instrument Problem

If instruments are only weakly correlated with the endogenous variable:
- 2SLS estimates are biased toward OLS
- Standard errors are too small
- Test statistics are unreliable

### Stock and Yogo (2005) Rule of Thumb

The first-stage F-statistic should exceed **10** to avoid serious weak instrument problems.



Work on the function `lm_2sls_ftest` that takes the arguments `Y` and `X_1`, `X_2`, `Z_1`, and `Z_2` and returns

* a scalar containing the first stage F-statistic for the restriction that the coefficients of $Z_2$ equal zero.

Once you have calculated the F-statistic, you can use Stock and Yogo's weak IV cutoffs to make a decision with regard to the strength/weakness of your instruments. What do you find?


### The F-Test

Here's a brief reminder about the first stage $F$-test (it generalized straightforwardly to any F-test in linear regression.) In the first stage you run this regression:

$$
X_{2i} = Z_{i1} \pi_1 + Z_{2i} \pi_2 + v_i
$$

You want to test if $\pi_2 = 0$ where $\dim(\pi_2) = L_2$ with $L_2$ possibly exceeding 1.

You have estimated $\pi$ by $\widehat{\pi}$ using OLS and you have estimated the covariance matrix so that

$$
\sqrt{N} (\widehat{\pi} - \pi) \overset{\text{approx}}{\sim} \mathcal{N} (0, \widehat{\Omega})
$$

Define the $L \times L_2$ dimensional matrix $R := (0_{L_2 \times K_1}, \quad I_{L_2})'$ for the purpose of selecting the appropriate elements of 
$\widehat{\pi} - \pi$. Then $R' (\widehat{\pi} - \pi) = \pi_2$. 

It follows that

$$
\sqrt{N} R' (\widehat{\pi} - \pi) \overset{\text{approx}}{\sim} \mathcal{N} (0, R' \widehat{\Omega} R).
$$

It follows, after the typical normal standardization, that for the quadratic form
$$
N (R' (\widehat{\pi} - \pi))' (R' \widehat{\Omega} R)^{-1} (R' (\widehat{\pi} - \pi))
\overset{\text{approx}}{\sim} \chi^2_{L_2}
$$

Our null hypothesis is that $\pi = 0$, consequently
$$
N (R' \widehat{\pi})' (R' \widehat{\Omega} R)^{-1} (R' \widehat{\pi})
\overset{\text{approx}}{\sim} \chi^2_{L_2}
$$

The F-statistic is the left hand side divided by $L_2$:
$$
F := 
\frac{N (R' \widehat{\pi})' (R' \widehat{\Omega} R)^{-1} (R' \widehat{\pi})}{L_2}
$$

(Notice: Hansen, in chapters 9.10 and 9.14, explains that for the purpose of calculating F one could use $\widehat{\Omega}$ that corresponds to the covariance estimator under *homoskedasticity*. Using the heteroskedasticity-robust version is ok here if you find it less confusing. The final test result happens to be unaffected.)

In [ ]:
# Compute F-statistic
F_stat = lm_2sls_ftest(Y, X1, X2, Z1, Z2_both)

@printf "\nFirst-Stage F-statistic: %.2f\n" F_stat

if F_stat > 10
    println("\nF > 10: Instruments appear sufficiently strong.")
else
    println("\nF < 10: Potential weak instrument problem!")
    println("  2SLS estimates may be biased toward OLS.")
    println("  Standard errors may be unreliable.")
end

## Summary Table

In [ ]:
println("\n" * "="^80 * "\n")
@printf "SUMMARY: ESTIMATES OF RETURN TO SCHOOLING (β2)\n"
println("="^80 * "\n\n")
@printf "%-25s %12s %12s %12s %12s\n" "Method" "Estimate" "Robust SE" "t-stat" "p-value"
println("-"^80 * "\n")
@printf "%-25s %12.3f %12.4f %12.2f %12.4f\n" "OLS" β_ols[2] se_ols[2] t_ols[2] p_ols[2]
@printf "%-25s %12.3f %12.4f %12.2f %12.4f\n" "IV (nearc4)" β_iv[end] se_iv[end] t_iv[end] p_iv[end]
@printf "%-25s %12.3f %12.4f %12.2f %12.4f\n" "2SLS (nearc2 + nearc4)" β_2sls[end] se_2sls[end] t_2sls[end] p_2sls[end]
println("-"^80 * "\n\n")
@printf "First-Stage F-statistic: %.2f\n" F_stat
@printf "(Stock-Yogo critical value: 10)\n"

## Summary and Interpretation

### Main Findings

1. **OLS estimate (7.5%)**: May be biased due to omitted ability

2. **IV/2SLS estimates (13-16%)**: Substantially higher than OLS!
   - This is **opposite** to what ability bias predicts
   - Possible explanations:
     - Measurement error in education (attenuates OLS toward zero)
     - Local Average Treatment Effect (LATE) interpretation
     - Instrument may not be fully valid

3. **Weak instruments**: F around 8 is below the Stock-Yogo threshold of 10
   - IV standard errors may be too small
   - Estimates may be biased toward OLS

### Econometric Lessons

1. **Always check first-stage strength**: F < 10 is a warning sign

2. **IV is not a free lunch**: Trades bias for variance
   - IV standard errors are much larger than OLS
   - Weak instruments can make this worse

3. **Think carefully about instrument validity**:
   - Relevance can be tested (F-statistic)
   - Exogeneity (exclusion restriction) cannot be directly tested